In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def main():
    print("Loading data...")
    df = pd.read_csv('htn_dat.csv')

    features = ['DBP', 'SBP', 'BMI', 'age', 'married', 'male.gender', 'hgb_centered',
                'adv_HIV', 'arv_naive', 'urban.clinic', 'log_creat_centered', 'SBP_ge120']
    target = 'event'

    X = df[features]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    preprocessor = SimpleImputer(strategy='median')

    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
    }

    os.makedirs('saved_models', exist_ok=True)

    metrics = {}
    print("Training models...")
    for name, model in models.items():
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        metrics[name] = {
            'Accuracy': round(accuracy_score(y_test, y_pred), 4),
            'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
            'Recall': round(recall_score(y_test, y_pred, zero_division=0), 4),
            'F1-Score': round(f1_score(y_test, y_pred, zero_division=0), 4),
            'ROC-AUC': round(roc_auc_score(y_test, y_prob), 4)
        }

        safe_name = name.replace(' ', '_')
        joblib.dump(pipeline, f'saved_models/{safe_name}_pipeline.joblib')
        print(f"  Saved {safe_name}_pipeline.joblib")

    # Save transformed training data for SHAP
    fitted_preprocessor = pipeline.named_steps['preprocessor']
    X_train_transformed = fitted_preprocessor.transform(X_train)

    joblib.dump(metrics, 'saved_models/metrics.joblib')
    joblib.dump(X_test, 'saved_models/X_test.joblib')
    joblib.dump(y_test, 'saved_models/y_test.joblib')
    joblib.dump(features, 'saved_models/features.joblib')
    joblib.dump(X_train_transformed, 'saved_models/X_train_transformed.joblib')

    print("\nAll artifacts saved to 'saved_models/' directory.")

if __name__ == "__main__":
    main()

Loading data...
Training models...
  Saved Logistic_Regression_pipeline.joblib
  Saved Random_Forest_pipeline.joblib
  Saved Gradient_Boosting_pipeline.joblib

All artifacts saved to 'saved_models/' directory.
